## Qwen3 fine-tuning on SQuARE — contrastive recipe

Fine-tunes **`Qwen/Qwen3-Embedding-0.6B`** on the **SQuARE** benchmark
(result-set-grounded SPARQL query pairs with answer-set Jaccard supervision),
reusing the SparqlCE main-branch training recipe from
`notebooks/training/train_dbpedia.ipynb`:

    loss      = InfoNCE(in-batch, tau=0.05) + 0.3 * MSE(cos(a,p), jaccard(a,p))
    optimizer = AdamW, cosine schedule with warmup, grad-clip 1.0
    selection = best val MRR (NOT mean cosine — that is biased toward collapse)
    splits    = paper's fixed split (splits_<kg>.json), seeded fallback otherwise

**What is revised vs. SparqlCE:**

| Component        | SparqlCE (CodeBERT + GATv2)                  | This notebook (Qwen3)                          |
|------------------|----------------------------------------------|------------------------------------------------|
| Input            | Query parsed into a graph (`SPARQLGraphBuilder`) | Raw SPARQL text, tokenized directly        |
| Encoder          | CodeBERT token emb + GATv2 + mean pool + proj | Qwen3-Embedding decoder, **last-token pooling** |
| Trainable params | Full model, differential LRs (1e-5 / 1e-4)    | **LoRA adapters** only, single LR 1e-4         |
| Precision        | fp32                                          | bf16 backbone + gradient checkpointing         |
| Augmentation     | Random variable renaming (graph-side)         | Same augmentation, reused on the text side     |

The loss, split handling, ranking-based validation, and checkpoint-selection
logic are carried over unchanged so the two models are directly comparable.

Run top-to-bottom. Verify the paths in the CONFIG cell first.


In [ ]:
# Dependencies (uncomment on a fresh environment)
# %pip install -q torch transformers>=4.51 peft accelerate tqdm


In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm

# ============================================================================
# 1. CONFIG — verify these paths match your repo before running
# ============================================================================
KG = "dbpedia"   # "dbpedia" | "wikidata"

PATHS = {
    "dbpedia": {
        "data":   "../../datasets/dbpedia-triplets-dataset.json",
        "splits": "../../datasets/splits_dbpedia.json",
    },
    "wikidata": {
        "data":   "../../datasets/wikidata-triplets-dataset.json",
        "splits": "../../datasets/splits_wikidata.json",
    },
}
DATA_PATH   = PATHS[KG]["data"]
SPLITS_PATH = PATHS[KG]["splits"]
CKPT_DIR    = f"../../checkpoints/qwen3_square_{KG}"

MODEL_NAME  = "Qwen/Qwen3-Embedding-0.6B"

# Recipe hyperparameters carried over from the SparqlCE training notebook
SEED         = 42
EPOCHS       = 5             # LoRA on a 0.6B model converges much faster than
                             # the from-scratch GNN (paper used 20 for SparqlCE)
TEMPERATURE  = 0.05          # paper: 0.05 (NOT 0.07)
LAMBDA_REG   = 0.3           # Jaccard calibration weight
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0

# Qwen3-specific settings
BATCH_SIZE   = 16            # per-step; in-batch negatives come from the batch
GRAD_ACCUM   = 2             # effective batch 32, matches the SparqlCE recipe
LR           = 1e-4          # single LR on LoRA params (backbone is frozen)
MAX_LEN      = 512           # tokens per SPARQL query
WARMUP_FRAC  = 0.05
AUGMENT_RENAME_PROB = 0.3    # same variable-renaming augmentation as SparqlCE

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]


In [ ]:
# ============================================================================
# 2. Setup: repo import path, seeds, device
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Reuse the variable-renaming augmentation from the SparqlCE dataset module so
# both models see the same invariance signal during training.
from sparqlce.dataset import _random_rename_variables  # noqa: E402

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")


### 3. Data

Same SQuARE JSON schema as `SemanticSPARQLDataset`
(`anchor_query` / `positive_queries` / `negative_queries`, each candidate
carrying its `answer_jaccard`), but the dataset below is **text-only**: no
graph builder, no PyG batching. In in-batch-negatives mode each item yields
`(anchor_text, positive_text, jaccard)` — negatives are the other batch items,
exactly as in the SparqlCE recipe.


In [ ]:
class SquareTextDataset(Dataset):
    """Anchor-level SQuARE dataset over raw SPARQL text.

    Text-side revision of ``sparqlce.dataset.SemanticSPARQLDataset``
    (in-batch-negatives mode): identical JSON parsing and augmentation,
    minus the graph construction.
    """

    def __init__(self, datapath, augment_rename_prob=0.0, augment_seed=None):
        self.augment_rename_prob = float(augment_rename_prob)
        self._aug_rng = random.Random(augment_seed) if augment_seed is not None else random
        self.groups = []

        print(f"Loading dataset from {datapath}...")
        with open(datapath, "r", encoding="utf-8") as f:
            raw = json.load(f)

        for item in raw:
            anchor = item.get("anchor_query")
            if not anchor:
                continue
            pos_list = [
                (p["query"], float(p["answer_jaccard"]))
                for p in item.get("positive_queries", [])
                if p.get("query") is not None and p.get("answer_jaccard") is not None
            ]
            neg_list = [
                (n["query"], float(n["answer_jaccard"]))
                for n in item.get("negative_queries", [])
                if n.get("query") is not None and n.get("answer_jaccard") is not None
            ]
            if not pos_list or not neg_list:
                continue
            self.groups.append({"anchor": anchor, "positives": pos_list})

        print(f"Loaded {len(self.groups)} anchor groups.")

    def _maybe_augment(self, query):
        if self.augment_rename_prob > 0.0 and self._aug_rng.random() < self.augment_rename_prob:
            return _random_rename_variables(query, self._aug_rng)
        return query

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        pos_q, pos_j = random.choice(g["positives"])
        return self._maybe_augment(g["anchor"]), self._maybe_augment(pos_q), float(pos_j)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")

def collate_pairs(batch):
    """Tokenize (anchor, positive, jaccard) triples into two padded batches."""
    anchors, positives, pos_js = zip(*batch)
    a_tok = tokenizer(list(anchors),   padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    p_tok = tokenizer(list(positives), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    return a_tok, p_tok, torch.tensor(pos_js, dtype=torch.float32)


In [ ]:
# ============================================================================
# 3b. Fixed split (paper) with safe random fallback — unchanged from SparqlCE
# ============================================================================
dataset = SquareTextDataset(
    DATA_PATH,
    augment_rename_prob=AUGMENT_RENAME_PROB,
    augment_seed=SEED,
)
# Validation/test must see un-augmented queries: share the file, not the flags.
eval_dataset = SquareTextDataset(DATA_PATH, augment_rename_prob=0.0)

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)
    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx   = splits.get("val_idx",   splits.get("valid_idx", splits.get("val")))
    test_idx  = splits.get("test_idx",  splits.get("test"))
    assert train_idx and val_idx and test_idx, f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset,      train_idx)
    val_dataset   = Subset(eval_dataset, val_idx)
    test_dataset  = Subset(eval_dataset, test_idx)
    print(f"Using FIXED split from {SPLITS_PATH}")
else:
    print(f"WARNING: fixed split file not found at {SPLITS_PATH}")
    print("WARNING: falling back to seeded random_split — numbers will NOT match the paper.")
    total   = len(dataset)
    n_train = int(0.80 * total)
    n_val   = int(0.10 * total)
    n_test  = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_split, val_split, test_split = random_split(
        range(total), [n_train, n_val, n_test], generator=gen)
    train_dataset = Subset(dataset,      list(train_split.indices))
    val_dataset   = Subset(eval_dataset, list(val_split.indices))
    test_dataset  = Subset(eval_dataset, list(test_split.indices))

print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_pairs, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)


### 4. Model: Qwen3-Embedding + LoRA

Qwen3-Embedding is a decoder-only model: the query embedding is the hidden
state of the **last non-padding token** (the pooling recommended on the model
card), L2-normalised — this replaces SparqlCE's GATv2 + mean-pool + projection
head. Only LoRA adapters are trained; the 0.6B backbone stays frozen in bf16.


In [ ]:
def last_token_pool(last_hidden_states, attention_mask):
    """Last-token pooling as on the Qwen3-Embedding model card.
    Handles both left- and right-padded batches."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device)
    return last_hidden_states[idx, seq_lens]


class Qwen3SparqlEmbedder(nn.Module):
    """Qwen3-Embedding backbone + LoRA, producing L2-normalised embeddings.

    Same output contract as ``HybridSPARQLEncoder.forward`` (unit-norm float
    embeddings), so the SparqlCE loss and evaluation code apply verbatim.
    """

    def __init__(self, model_name, lora_cfg=None, adapter_path=None):
        super().__init__()
        base = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})
        base.enable_input_require_grads()  # needed for checkpointing + frozen base
        if adapter_path is not None:
            self.backbone = PeftModel.from_pretrained(base, adapter_path)
        else:
            self.backbone = get_peft_model(base, lora_cfg)

    def forward(self, tok):
        out = self.backbone(input_ids=tok["input_ids"],
                            attention_mask=tok["attention_mask"])
        emb = last_token_pool(out.last_hidden_state, tok["attention_mask"])
        return F.normalize(emb.float(), p=2, dim=1)


lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="FEATURE_EXTRACTION",
)

model = Qwen3SparqlEmbedder(MODEL_NAME, lora_cfg=lora_cfg).to(device)
model.backbone.print_trainable_parameters()

# Single LR on the LoRA params replaces SparqlCE's differential-LR scheme:
# there is no random-init GNN here, and the backbone is frozen.
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)
steps_per_epoch = len(train_loader) // GRAD_ACCUM
total_steps     = steps_per_epoch * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)


In [ ]:
# ============================================================================
# 5. Losses — identical to the SparqlCE recipe
# ============================================================================
def infonce_inbatch(a_emb, p_emb, temperature):
    """In-batch InfoNCE. Matches the paper / main branch."""
    a = F.normalize(a_emb, dim=1)
    p = F.normalize(p_emb, dim=1)
    logits = a @ p.T / temperature
    labels = torch.arange(a.size(0), device=a.device)
    return F.cross_entropy(logits, labels)


In [ ]:
# ============================================================================
# 6. Validation: rank each anchor's true positive against ALL positives in
# the split. Identical to SparqlCE — selects the embedding that retrieves
# correctly, not the one that collapses everything to a single point.
# ============================================================================
@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []
    for a_tok, p_tok, _ in loader:
        a_tok = {k: v.to(device) for k, v in a_tok.items()}
        p_tok = {k: v.to(device) for k, v in p_tok.items()}
        anchors.append(model(a_tok).cpu())
        positives.append(model(p_tok).cpu())
    A = torch.cat(anchors)      # [N, d]
    P = torch.cat(positives)    # [N, d]
    sims  = A @ P.T             # [N, N]; diagonal is the true positive
    diag  = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()   # 1-indexed rank of true positive
    return {
        "mrr":    (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }


In [ ]:
# ============================================================================
# 7. Training loop — SparqlCE recipe with gradient accumulation
# ============================================================================
os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr  = -1.0
best_path = os.path.join(CKPT_DIR, "best_adapter")
history   = []

print(f"Training Qwen3 on SQuARE-{KG} for {EPOCHS} epochs "
      f"(effective batch {BATCH_SIZE * GRAD_ACCUM})")
print(f"loss = InfoNCE(tau={TEMPERATURE}) + {LAMBDA_REG} * MSE(cos, jaccard)")

for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss, sum_ce, sum_reg, n_steps = 0.0, 0.0, 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")

    for step, (a_tok, p_tok, pos_js) in enumerate(pbar, start=1):
        a_tok  = {k: v.to(device) for k, v in a_tok.items()}
        p_tok  = {k: v.to(device) for k, v in p_tok.items()}
        pos_js = pos_js.to(device)

        a_emb = model(a_tok)
        p_emb = model(p_tok)

        # InfoNCE on the batch (in-batch negatives)
        loss_ce = infonce_inbatch(a_emb, p_emb, TEMPERATURE)

        # Jaccard calibration MSE on positives (embeddings already unit-norm)
        cos = (a_emb * p_emb).sum(dim=1)
        loss_reg = (cos - pos_js).pow(2).mean()

        loss = loss_ce + LAMBDA_REG * loss_reg
        (loss / GRAD_ACCUM).backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        sum_loss += loss.item(); sum_ce += loss_ce.item(); sum_reg += loss_reg.item()
        n_steps  += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         ce=f"{loss_ce.item():.4f}",
                         reg=f"{loss_reg.item():.4f}")

    val_metrics = evaluate_ranking(model, val_loader, device)
    history.append({"epoch": epoch,
                    "train_loss": sum_loss / n_steps,
                    **{f"val_{k}": v for k, v in val_metrics.items()}})
    print(f"Epoch {epoch:02d} | loss {sum_loss/n_steps:.4f} "
          f"(ce {sum_ce/n_steps:.4f}, reg {sum_reg/n_steps:.4f}) | "
          f"val MRR {val_metrics['mrr']:.4f} "
          f"H@1 {val_metrics['hits@1']:.4f} H@5 {val_metrics['hits@5']:.4f}")

    # Selection by val MRR, as in the SparqlCE recipe. Only the LoRA adapter
    # is saved (~a few MB) instead of a full state dict.
    if val_metrics["mrr"] > best_mrr:
        best_mrr = val_metrics["mrr"]
        model.backbone.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        print(f"  ✓ new best val MRR {best_mrr:.4f} — adapter saved to {best_path}")


In [ ]:
# ============================================================================
# 8. Final evaluation on the test split
# ============================================================================
print(f"Training complete. Best val MRR: {best_mrr:.4f}")
print("Reloading best adapter and evaluating on the test split...")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_model = Qwen3SparqlEmbedder(MODEL_NAME, adapter_path=best_path).to(device)
test_metrics = evaluate_ranking(best_model, test_loader, device)

print(f"\nSQuARE-{KG} test results (fine-tuned Qwen3-Embedding-0.6B):")
for k, v in test_metrics.items():
    print(f"  {k:>7}: {v:.4f}")

with open(os.path.join(CKPT_DIR, "test_metrics.json"), "w") as f:
    json.dump({"kg": KG, "model": MODEL_NAME, "best_val_mrr": best_mrr,
               "test": test_metrics, "history": history}, f, indent=2)
print(f"\nMetrics written to {os.path.join(CKPT_DIR, 'test_metrics.json')}")


### Notes

- **Comparability.** Loss, temperature, calibration weight, fixed splits, and
  the MRR-based selection are identical to the SparqlCE training notebook, so
  the resulting numbers slot directly into the paper's baseline tables as the
  *fine-tuned* Qwen3 row (vs. the zero-shot Qwen3 baseline).
- **Zero-shot reference.** To get the zero-shot number with the same eval,
  skip section 7 and run `evaluate_ranking` on a fresh
  `Qwen3SparqlEmbedder(MODEL_NAME, lora_cfg=lora_cfg)` — the untrained LoRA
  is an identity at init (`lora_B = 0`), so it reproduces the base model.
- **Wikidata.** Set `KG = "wikidata"` in the CONFIG cell; the notebook expects
  `datasets/wikidata-triplets-dataset.json` and `datasets/splits_wikidata.json`.
- **Memory.** ~0.6B params in bf16 + LoRA + checkpointing fits comfortably on a
  single 24 GB GPU at batch 16 / 512 tokens. Reduce `BATCH_SIZE` and raise
  `GRAD_ACCUM` proportionally if you hit OOM — the loss uses in-batch
  negatives, so the *per-step* batch (not effective batch) sets negative count.
